# Live AWBC: cross-fitted Value + frozen-backbone Actor
Preparation and offline training only. No new runtime is allocated. Start on CPU to copy data; select a GPU for training.
Upload `crrl_live_ppo_colab.zip`, `live_awbc_update.zip`, and completed session directories to `MyDrive/cr_rl_live_ppo/`. Every training session must have its own verified `result.json`.
The first version fits only the Critic value head and Actor card/WAIT/ability readouts. Spatial and recurrent backbones remain frozen. Low-quality out-of-fold Value triggers plain-BC fallback, clearly reported in logs. This does not prove game strength.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive', force_remount=True)
DRIVE = Path('/content/drive/MyDrive/cr_rl_live_ppo')
ROOT = Path('/content/crrl_live_ppo')
BASE = DRIVE / 'crrl_live_ppo_colab.zip'
PATCH = DRIVE / 'live_awbc_update.zip'
RUN_NAME = 'awbc_readouts_v1'
assert BASE.is_file() and PATCH.is_file(), 'Upload the base checkpoint ZIP and AWBC code update first'


In [ ]:
import hashlib, json, os, shutil, zipfile
from tqdm.auto import tqdm
ROOT.mkdir(parents=True, exist_ok=True)
for bundle in (BASE, PATCH):
    with zipfile.ZipFile(bundle) as archive:
        manifest = json.loads(archive.read('BUNDLE_MANIFEST.json'))
        assert isinstance(manifest, list), 'Use the 4.38GB live PPO bundle, not the older multi-opponent bundle'
        missing = [row for row in manifest if not (ROOT / row['path']).exists()]
        assert shutil.disk_usage(ROOT).free > sum(row['bytes'] for row in missing) + 2*1024**3
        for row in tqdm(manifest, desc=bundle.name):
            target = (ROOT / row['path']).resolve()
            assert target.is_relative_to(ROOT.resolve()), row['path']
            if target.exists() and target.stat().st_size == row['bytes']:
                with target.open('rb') as stream:
                    if hashlib.file_digest(stream, 'sha256').hexdigest() == row['sha256']: continue
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_suffix(target.suffix + '.partial')
            digest = hashlib.sha256()
            with archive.open(row['path']) as src, temporary.open('wb') as dst:
                for block in iter(lambda: src.read(4*1024**2), b''):
                    digest.update(block); dst.write(block)
            assert digest.hexdigest() == row['sha256'] and temporary.stat().st_size == row['bytes']
            os.replace(temporary, target)
source = DRIVE / 'sessions'
assert source.is_dir(), 'Upload session folders into sessions/'
# Copy every recorded file so strict provenance and execution checks remain available.
for folder in tqdm(sorted(p for p in source.iterdir() if p.is_dir()), desc='Copy session folders'):
    for path in folder.rglob('*'):
        if not path.is_file(): continue
        target = ROOT / 'sessions' / folder.name / path.relative_to(folder)
        target.parent.mkdir(parents=True, exist_ok=True)
        # Input sessions must be finalized before upload.
        with path.open('rb') as stream: expected = hashlib.file_digest(stream, 'sha256').hexdigest()
        if target.exists():
            with target.open('rb') as stream:
                if hashlib.file_digest(stream, 'sha256').hexdigest() == expected: continue
        temporary = target.with_suffix(target.suffix + '.partial')
        shutil.copyfile(path, temporary)
        with temporary.open('rb') as stream: assert hashlib.file_digest(stream, 'sha256').hexdigest() == expected
        os.replace(temporary, target)
print('Local files verified. No training has started.')


In [ ]:
import importlib.util, subprocess, sys
assert importlib.util.find_spec('torch'), 'Use Colab with preinstalled Torch'
requirements = {'numpy':'numpy>=2', 'tqdm':'tqdm>=4.66', 'yaml':'PyYAML>=6'}
missing = [package for name,package in requirements.items() if importlib.util.find_spec(name) is None]
if missing: subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Optional W&B: API key comes from Colab Secrets, never from the ZIP.
USE_WANDB = True
if USE_WANDB:
    from google.colab import userdata
    if not os.environ.get('WANDB_API_KEY'):
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    if importlib.util.find_spec('wandb') is None:
        subprocess.run([sys.executable,'-m','pip','install','wandb'],check=True)
WORK = ROOT / 'artifacts' / RUN_NAME
BACKUP = DRIVE / 'runs' / RUN_NAME
command = [sys.executable,'scripts/train_live_awbc.py','--sessions',str(ROOT/'sessions'),
           '--checkpoint',str(ROOT/'ppo_latest.pt'),'--out',str(WORK),
           '--device','cuda','--epochs','8','--value-epochs','50',
           '--actor-lr','1e-6','--value-lr','1e-4','--strength','0.5','--beta','0.5']
if USE_WANDB: command += ['--wandb']
# Child output is kept locally; the supervisor shows progress and hash-verifies Drive backups.
subprocess.run([sys.executable,'scripts/run_live_awbc_colab.py','--work',str(WORK),
                '--backup',str(BACKUP),'--',*command],check=True)


Check `excluded_sessions.json`, `dataset.json`, `value_crossfit.json`, `fixed_weights.npz`, and `metrics.jsonl`. At least 12 qualified games are required. `value_gate_passed=false` means no evidence for usable Value weighting: the run uses ordinary BC weights. `best.pt` includes epoch zero as a fallback; offline NLL/KL are guardrails, not a win-rate evaluation.
Rerun with the same finalized sessions/settings/output to resume `latest.pt`; use a new RUN_NAME if data or settings change. The new Actor export is independent of the fitted Value head; the latter is stored in `value_head.pt` and requires the original Critic backbone. No simulator is required for this offline training.